# 08 — Flight Distance experiments (LightGBM only)

**Objective:** Find out whether treating `Flight Distance` as a repeated route identifier improves LightGBM **and**, more importantly, your current **70% tree ensemble + 30% RealMLP** blend.

This notebook tests four LightGBMs on the **same outer 5 folds** as Notebook 04:

| Variant | Added feature(s) |
|---|---|
| `control` | No new features: reproduce Notebook 04's LightGBM |
| `count` | Frequency of each Flight Distance |
| `target` | Smoothed Flight Distance target encoding, **cross-fitted within each outer training fold** |
| `both` | Count + target encoding |

**What stays untouched:** your original notebooks, original tree and RealMLP OOF predictions, and your best Kaggle submission. No CatBoost, XGBoost, or RealMLP training is performed here.

**Important:** This is an **OOF experiment, not a new Kaggle submission**. Because Notebook 04 did not save *individual* LightGBM test predictions, we cannot exactly reconstruct the original test ensemble by swapping its LightGBM component. This experiment nevertheless saves the newly trained control and candidate test predictions (only if the candidate improves OOF) for a later, separately validated submission step.

> Required packages: `numpy`, `pandas`, `scikit-learn`, `lightgbm`, `IPython`. Run top-to-bottom with the project environment you used for Notebook 04.

In [ ]:
from pathlib import Path
import gc
import time
import numpy as np
import pandas as pd
from IPython.display import display
from lightgbm import LGBMClassifier, early_stopping
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score

# ---- Controls: start with SMOKE_TEST=True if you want a quick wiring check. ----
SMOKE_TEST = False                # Full comparison requires False (all original rows, 5 folds).
THREADS = 4                      # Same setting as Notebook 04.
SEED = 42                        # Same outer folds as Notebooks 04 and 07.
INNER_FOLDS = 5                  # Only for leakage-safe training target encodings.
TARGET_SMOOTHING = 50.0          # Fixed in advance; do not tune on each validation fold.
VARIANTS = ("control", "count", "target", "both")


def find_project_root():
    """Works when Jupyter opens in project root OR project_root/notebooks."""
    for start in (Path.cwd().resolve(), Path.cwd().resolve().parent):
        for candidate in (start, *start.parents):
            if all((candidate / "data" / "raw" / name).exists()
                   for name in ("train.csv", "test.csv", "sample_submission.csv")):
                return candidate
    raise FileNotFoundError(
        "Project root not found. Expected data/raw/train.csv, "
        "data/raw/test.csv, and data/raw/sample_submission.csv."
    )


ROOT = find_project_root()
DATA_DIR = ROOT / "data" / "raw"
SUB_DIR = ROOT / "submissions"
OUTPUT_DIR = SUB_DIR / "exp08_flight_distance"

full_train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

assert full_train["id"].is_unique and test["id"].is_unique
assert test["id"].equals(sample_submission["id"]), "Test IDs/order mismatch."
assert "satisfaction" in full_train and "satisfaction" not in test

if SMOKE_TEST:
    # Smoke-only subset; scores are NOT comparable to the full-data saved OOF baseline.
    train, _ = train_test_split(
        full_train, train_size=min(20_000, int(0.7 * len(full_train))),
        stratify=full_train["satisfaction"], random_state=SEED,
    )
    train = train.reset_index(drop=True)
    n_folds, n_estimators, stopping_rounds, inner_folds = 2, 120, 20, 3
else:
    train = full_train
    n_folds, n_estimators, stopping_rounds, inner_folds = 5, 1400, 90, INNER_FOLDS

y = train["satisfaction"].astype(int).to_numpy()
features = [col for col in train if col not in ("id", "satisfaction")]
assert set(features) == set(test.columns) - {"id"}
assert "Flight Distance" in features

# Match Notebook 04 exactly: fill string categories, one-hot encode, keep numeric NaN.
X_raw = train[features].copy()
T_raw = test[features].copy()
cat_cols = X_raw.select_dtypes(include=["object", "category", "string"]).columns.tolist()
for col in cat_cols:
    X_raw[col] = X_raw[col].fillna("Missing").astype(str)
    T_raw[col] = T_raw[col].fillna("Missing").astype(str)

X_base = pd.get_dummies(X_raw, columns=cat_cols, dtype=np.int8)
T_base = pd.get_dummies(T_raw, columns=cat_cols, dtype=np.int8)
T_base = T_base.reindex(columns=X_base.columns, fill_value=0)

splits = list(StratifiedKFold(
    n_splits=n_folds, shuffle=True, random_state=SEED
).split(X_base, y))
fold_of_row = np.zeros(len(train), dtype=np.int8)
for fold, (_, va_idx) in enumerate(splits, start=1):
    fold_of_row[va_idx] = fold

print("Root:", ROOT)
print("Mode:", "SMOKE TEST — NO BASELINE COMPARISON OR SAVED OUTPUT" if SMOKE_TEST else "FULL 5-FOLD COMPARISON")
print(f"Train={len(train):,}; test={len(test):,}; original encoded columns={X_base.shape[1]}")
print("Features for this experiment: count; target; both; unchanged control")

## Load and verify the frozen OOF baseline

We align **by `id`**, not by assumed CSV row order. We also check labels and (where available) RealMLP's fold assignments. If any necessary baseline file is missing, this notebook fails **before expensive training starts**.

Your saved blend used tree weights `CatBoost=0.25`, `XGBoost=0.25`, `LightGBM=0.50` and final weights **70% trees / 30% RealMLP**.

In [ ]:
def find_existing_file(candidates, explanation):
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(
        f"Missing {explanation}. Searched:\n" + "\n".join(str(p) for p in candidates)
    )


def aligned_oof(path, needed_columns):
    df = pd.read_csv(path)
    if "id" not in df or "satisfaction" not in df or not df["id"].is_unique:
        raise ValueError(f"Invalid OOF file (missing/duplicate IDs or labels): {path}")
    required = ["satisfaction"] + list(needed_columns)
    if not set(required).issubset(df.columns):
        raise ValueError(f"Missing columns from {path}: {set(required) - set(df.columns)}")
    index = pd.Index(train["id"].to_numpy())
    if not index.isin(df["id"]).all():
        raise ValueError(f"OOF IDs do not cover train.csv: {path}")
    aligned = df.set_index("id").reindex(index)
    if aligned[required].isna().any().any():
        raise ValueError(f"Missing prediction/label in {path}")
    if not np.array_equal(aligned["satisfaction"].to_numpy(dtype=int), y):
        raise ValueError(f"OOF labels disagree with train.csv: {path}")
    return aligned


if not SMOKE_TEST:
    tree_path = find_existing_file(
        [SUB_DIR / "04_oof_predictions.csv"],
        "the original Notebook 04 OOF file",
    )
    realmlp_path = find_existing_file(
        [
            SUB_DIR / "realmlp_submission_v1" / "paired_oof_predictions.csv",
            SUB_DIR / "exp07_realmlp" / "full" / "paired_oof_predictions.csv",
            ROOT / "realmlp_submission_v1" / "paired_oof_predictions.csv",
        ],
        "the original RealMLP OOF file",
    )

    original_tree = aligned_oof(
        tree_path, ["oof_CatBoost", "oof_XGBoost", "oof_LightGBM"]
    )
    original_realmlp = aligned_oof(realmlp_path, ["oof_RealMLP", "fold"])
    if not np.array_equal(original_realmlp["fold"].to_numpy(dtype=int), fold_of_row):
        raise ValueError("RealMLP OOF used different folds. Do not combine these predictions.")

    p_cb = original_tree["oof_CatBoost"].to_numpy(dtype=float)
    p_xgb = original_tree["oof_XGBoost"].to_numpy(dtype=float)
    p_lgb_original = original_tree["oof_LightGBM"].to_numpy(dtype=float)
    p_realmlp = original_realmlp["oof_RealMLP"].to_numpy(dtype=float)

    original_tree_blend = 0.25 * p_cb + 0.25 * p_xgb + 0.50 * p_lgb_original
    original_final_blend = 0.70 * original_tree_blend + 0.30 * p_realmlp
    original_lgb_auc = roc_auc_score(y, p_lgb_original)
    original_final_auc = roc_auc_score(y, original_final_blend)

    print("Loaded:", tree_path)
    print("Loaded:", realmlp_path)
    print(f"Original LightGBM OOF AUC: {original_lgb_auc:.8f}")
    print(f"Original 70/30 blend OOF AUC: {original_final_auc:.8f}")
    # Warnings (not rigid equality): saved float formatting may change last digits.
    if abs(original_lgb_auc - 0.958810) > 0.0002 or abs(original_final_auc - 0.959059) > 0.0002:
        raise ValueError("Saved baselines disagree with your known OOF scores; verify files before training.")
else:
    print("Smoke mode: not loading or comparing original full-data OOF predictions.")

## Leakage-safe route features

- **Count encoding**: value counts fitted on the **outer training rows only**, mapped to training, validation, and test.
- **Target encoding**: for the **outer training rows**, compute each encoded value from one of five **inner training folds that excludes that row**. For outer validation/test, fit the encoding using all outer training labels only. Unseen distances map to the training label mean.
- **Smoothing**: `(positive_count + 50 × training_prior) / (distance_count + 50)`. This shrinks rare distances toward the fold's satisfaction rate.

`Flight Distance` remains as its original numeric feature in every model; these are additional columns. No validation labels are ever used to construct that validation fold's target encoding.

In [ ]:
def distance_keys(frame):
    """Treat each exact observed numeric Flight Distance as a category/group."""
    distance = pd.to_numeric(frame["Flight Distance"], errors="raise").astype("float64")
    if distance.dropna().lt(0).any():
        raise ValueError("Flight Distance has negative values; revise missing-distance sentinel.")
    return distance.fillna(-1.0).reset_index(drop=True)


def smoothed_target_map(keys, targets, smoothing):
    prior = float(np.mean(targets))
    grouped = pd.DataFrame({"key": np.asarray(keys), "target": np.asarray(targets)}) \
        .groupby("key", sort=False)["target"].agg(["sum", "count"])
    rate = (grouped["sum"] + smoothing * prior) / (grouped["count"] + smoothing)
    return rate, prior


def features_for_fold(train_idx, val_idx, fold):
    """Fit route preprocessing within outer train; cross-fit TRAIN target encoding."""
    tr_keys = distance_keys(X_raw.iloc[train_idx])
    va_keys = distance_keys(X_raw.iloc[val_idx])
    te_keys = distance_keys(T_raw)
    tr_y = y[train_idx]

    # Supervised/unsupervised features are created once and reused across variants.
    counts = tr_keys.value_counts(dropna=False)
    count_tr = tr_keys.map(counts).to_numpy(dtype=np.float32)
    count_va = va_keys.map(counts).fillna(0).to_numpy(dtype=np.float32)
    count_te = te_keys.map(counts).fillna(0).to_numpy(dtype=np.float32)

    inner_splitter = StratifiedKFold(
        n_splits=inner_folds, shuffle=True, random_state=SEED + fold
    )
    target_tr = np.full(len(train_idx), np.nan, dtype=np.float32)
    for inner_fit, inner_holdout in inner_splitter.split(np.zeros(len(train_idx)), tr_y):
        encoding, prior = smoothed_target_map(
            tr_keys.iloc[inner_fit], tr_y[inner_fit], TARGET_SMOOTHING
        )
        target_tr[inner_holdout] = (
            tr_keys.iloc[inner_holdout].map(encoding).fillna(prior).to_numpy(dtype=np.float32)
        )

    full_encoding, full_prior = smoothed_target_map(tr_keys, tr_y, TARGET_SMOOTHING)
    target_va = va_keys.map(full_encoding).fillna(full_prior).to_numpy(dtype=np.float32)
    target_te = te_keys.map(full_encoding).fillna(full_prior).to_numpy(dtype=np.float32)
    assert np.isfinite(target_tr).all() and np.isfinite(target_va).all()
    assert ((target_tr >= 0) & (target_tr <= 1)).all()
    assert ((target_va >= 0) & (target_va <= 1)).all()

    return {
        "count": (count_tr, count_va, count_te),
        "target": (target_tr, target_va, target_te),
    }


def make_fold_data(indices, route_features, variant, dataset):
    """Return unchanged Notebook 04 design matrix plus requested route columns."""
    if dataset == "train":
        result = X_base.iloc[indices].copy()
        feature_part = 0
    elif dataset == "validation":
        result = X_base.iloc[indices].copy()
        feature_part = 1
    elif dataset == "test":
        result = T_base.copy()
        feature_part = 2
    else:
        raise ValueError(dataset)

    if variant in ("count", "both"):
        result["FD_route_count"] = route_features["count"][feature_part]
    if variant in ("target", "both"):
        result["FD_route_target_mean"] = route_features["target"][feature_part]
    return result


def make_lightgbm():
    """Exactly the hyperparameters and early-stopping protocol of Notebook 04."""
    return LGBMClassifier(
        n_estimators=n_estimators,
        learning_rate=0.045,
        num_leaves=31,
        min_child_samples=40,
        subsample=0.85,
        subsample_freq=1,
        colsample_bytree=0.9,
        reg_lambda=2.0,
        objective="binary",
        verbosity=-1,
        random_state=SEED,
        n_jobs=THREADS,
    )

## Train the four LightGBMs

This is the only expensive cell. A full run fits **4 variants × 5 outer folds = 20 LightGBM models** (plus inexpensive inner target-encoding calculations). Predictions are stored **in memory** until the end; no checkpoint clutter is created.

Test predictions are also averaged across folds during the full run, so we won't need to retrain the same models just to obtain test probabilities later. This cell **does not** create a Kaggle submission.

In [ ]:
oof_by_variant = {
    name: np.full(len(train), np.nan, dtype=np.float64) for name in VARIANTS
}
test_sum_by_variant = {
    name: np.zeros(len(test), dtype=np.float64) for name in VARIANTS
} if not SMOKE_TEST else {}
fold_records = []
run_start = time.perf_counter()

for fold, (tr_idx, va_idx) in enumerate(splits, start=1):
    print(f"\n===== Fold {fold}/{n_folds} =====", flush=True)
    route_features = features_for_fold(tr_idx, va_idx, fold)

    for variant in VARIANTS:
        start = time.perf_counter()
        x_tr = make_fold_data(tr_idx, route_features, variant, "train")
        x_va = make_fold_data(va_idx, route_features, variant, "validation")

        model = make_lightgbm()
        model.fit(
            x_tr, y[tr_idx],
            eval_set=[(x_va, y[va_idx])],
            eval_metric="auc",
            callbacks=[early_stopping(stopping_rounds, verbose=False)],
        )
        valid_pred = model.predict_proba(x_va)[:, 1]
        oof_by_variant[variant][va_idx] = valid_pred
        fold_auc = roc_auc_score(y[va_idx], valid_pred)

        if not SMOKE_TEST:
            x_test = make_fold_data(None, route_features, variant, "test")
            test_sum_by_variant[variant] += model.predict_proba(x_test)[:, 1] / n_folds
            del x_test

        fold_records.append({
            "fold": fold,
            "variant": variant,
            "lightgbm_fold_auc": fold_auc,
            "best_iteration": model.best_iteration_,
            "seconds": round(time.perf_counter() - start, 2),
        })
        print(f"  {variant:8s} | AUC {fold_auc:.7f} | "
              f"best iteration {model.best_iteration_} | "
              f"{time.perf_counter() - start:.0f}s", flush=True)
        del model, x_tr, x_va
    del route_features
    gc.collect()

for variant in VARIANTS:
    assert np.isfinite(oof_by_variant[variant]).all(), f"Missing OOF predictions: {variant}"
    if not SMOKE_TEST:
        p = test_sum_by_variant[variant]
        assert np.isfinite(p).all() and np.all((p >= 0) & (p <= 1))

print(f"\nAll experiments complete in {(time.perf_counter() - run_start)/60:.1f} minutes.")

## Compare LightGBM **and** the real full blend

For each new LightGBM prediction vector, we substitute it into your **saved, independently trained** CatBoost, XGBoost, and RealMLP OOF predictions:

\[
\mathrm{NewBlend} = 0.7(0.25\,CB + 0.25\,XGB + 0.50\,LGB_{new}) + 0.3\,RealMLP.
\]

This is a **what-if blend using the existing models**, not the result of retraining all ensemble members. We track both overall AUC and the AUC change on every original outer validation fold.

The retrained `control` is also compared against the saved original LightGBM. If they differ, the discrepancy may reflect library or environment changes—not a feature-engineering gain.

In [ ]:
fold_df = pd.DataFrame(fold_records)
rows = []
blend_by_variant = {}
for variant in VARIANTS:
    lgb_pred = oof_by_variant[variant]
    item = {"variant": variant, "lightgbm_oof_auc": roc_auc_score(y, lgb_pred)}
    if not SMOKE_TEST:
        candidate_blend = 0.70 * (0.25 * p_cb + 0.25 * p_xgb + 0.50 * lgb_pred) + 0.30 * p_realmlp
        blend_by_variant[variant] = candidate_blend
        item["full_blend_oof_auc"] = roc_auc_score(y, candidate_blend)
        item["delta_vs_original_lightgbm"] = item["lightgbm_oof_auc"] - original_lgb_auc
        item["delta_vs_original_full_blend"] = item["full_blend_oof_auc"] - original_final_auc
        for fold, (_, va_idx) in enumerate(splits, start=1):
            item[f"fold_{fold}_blend_delta"] = (
                roc_auc_score(y[va_idx], candidate_blend[va_idx])
                - roc_auc_score(y[va_idx], original_final_blend[va_idx])
            )
    rows.append(item)

results = pd.DataFrame(rows)
if not SMOKE_TEST:
    results = results.sort_values("full_blend_oof_auc", ascending=False).reset_index(drop=True)
    control_diff_auc = roc_auc_score(y, oof_by_variant["control"]) - original_lgb_auc
    control_mae = np.mean(np.abs(oof_by_variant["control"] - p_lgb_original))
    print(f"Saved-original versus refitted-control LightGBM | AUC delta={control_diff_auc:+.8f} "
          f"| OOF prediction MAE={control_mae:.8f}")
    if abs(control_diff_auc) > 0.0001 or control_mae > 0.001:
        print("WARNING: baseline reproduction differs noticeably. Check package versions / preprocessing; "
              "compare all feature variants to the refitted control as well.")
    results["delta_vs_refitted_control_lightgbm"] = (
        results["lightgbm_oof_auc"] - roc_auc_score(y, oof_by_variant["control"])
    )
    results["delta_vs_refitted_control_blend"] = (
        results["full_blend_oof_auc"] - roc_auc_score(y, blend_by_variant["control"])
    )
else:
    results = results.sort_values("lightgbm_oof_auc", ascending=False).reset_index(drop=True)

display(results.round(7))
print("\nLightGBM fold AUCs:")
display(fold_df.pivot(index="fold", columns="variant", values="lightgbm_fold_auc").round(6))

if not SMOKE_TEST:
    changed = results.loc[results["variant"] != "control"].copy()
    best = changed.sort_values("full_blend_oof_auc", ascending=False).iloc[0]
    print(f"\nBest engineered variant by full OOF blend: {best['variant']} ")
    print(f"New full blend: {best['full_blend_oof_auc']:.8f}")
    print(f"Original full blend: {original_final_auc:.8f}")
    print(f"OOF delta: {best['delta_vs_original_full_blend']:+.8f}")
    gains = [best[f"fold_{i}_blend_delta"] for i in range(1, n_folds + 1)]
    print(f"Positive fold deltas: {sum(v > 0 for v in gains)} / {n_folds}")
    print("Fold blend deltas:", [round(v, 7) for v in gains])
    if best["delta_vs_original_full_blend"] <= 0:
        print("Decision: no tested feature improved the saved full blend. Keep the current model.")
    else:
        print("Decision: promising OOF candidate. Test robustness before Kaggle submission.")

## Save **only** useful results

Full mode writes:

- `submissions/exp08_flight_distance/08_results.csv` — one small experiment comparison (including fold deltas).
- **Only if an engineered variant beats the original full blend OOF:**
  - `08_best_candidate_oof.csv` — IDs, labels, fold, improved LightGBM OOF, and full-blend OOF.
  - `08_best_candidate_test_components.csv` — test IDs, new LightGBM test probabilities, and matched `control` LightGBM test probabilities.

No `.npz` checkpoints, fold metadata JSONs, or extra submissions. Smoke mode writes **nothing**.

**Important:** Do not submit the candidate test LightGBM predictions as if they were the final blended submission. We'll use them later if the OOF results justify it. The original `07_blend30_submission.csv` remains untouched.

In [ ]:
if SMOKE_TEST:
    print("Smoke test finished; nothing saved. Set SMOKE_TEST=False and rerun all cells for real CV.")
else:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    results_path = OUTPUT_DIR / "08_results.csv"
    results.to_csv(results_path, index=False, float_format="%.10f")
    print("Saved:", results_path)

    if best["delta_vs_original_full_blend"] > 0:
        winner = str(best["variant"])
        oof_output = pd.DataFrame({
            "id": train["id"].to_numpy(),
            "satisfaction": y,
            "fold": fold_of_row,
            "oof_LightGBM_improved": oof_by_variant[winner],
            "oof_full_70_30_blend": blend_by_variant[winner],
        })
        oof_path = OUTPUT_DIR / "08_best_candidate_oof.csv"
        oof_output.to_csv(oof_path, index=False, float_format="%.10f")

        test_output = pd.DataFrame({
            "id": test["id"].to_numpy(),
            "control_LightGBM_test": test_sum_by_variant["control"],
            "improved_LightGBM_test": test_sum_by_variant[winner],
        })
        test_path = OUTPUT_DIR / "08_best_candidate_test_components.csv"
        test_output.to_csv(test_path, index=False, float_format="%.10f")
        print("Saved:", oof_path)
        print("Saved:", test_path)
        print("NOTE: These are candidate components, not a finished Kaggle submission.")
    else:
        print("No better full-blend OOF candidate: no additional predictions saved.")

## Interpreting the outcome

1. **First:** compare `count`, `target`, and `both` against the **refitted control** to see whether the additional features themselves helped under the current library versions.
2. **Second:** compare each **full-blend OOF AUC** to the original full-blend score of approximately `0.959059`.
3. **Third:** look at the five **fold deltas**. A tiny overall gain in only one fold may be noise; a repeatable gain on most folds is more compelling.
4. **Only after a verified improvement:** try the winning features on XGBoost / RealMLP, and decide how to construct a **correct** new test submission. Do not use the public leaderboard to pick among many noisy experiments.

**Why not try 15 folds or 100+ features here?** Keeping one hypothesis and fixed settings makes the result easier to interpret.